In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.linear_model import Lasso

from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error as MSE, accuracy_score
from sklearn.ensemble import RandomForestRegressor
from sklearn.tree import DecisionTreeClassifier
from scipy.stats import pearsonr
from sklearn.model_selection import RandomizedSearchCV


In [ ]:
original_df = pd.read_csv('rental_info.csv')

original_df['rental_date']  = pd.to_datetime ( original_df['rental_date'] , errors = 'coerce' )
original_df['return_date']  = pd.to_datetime ( original_df['return_date'] , errors = 'coerce' )

rental_seconds  = (original_df['return_date']-original_df['rental_date']).dt.total_seconds()
original_df['rental_length_days'] = np.round(rental_seconds/(24*3600),1)

print (original_df.head())
print (original_df.info())
print (original_df.describe())

In [ ]:
counts = original_df['rental_length_days'].value_counts().sort_index()

print(counts)

plt.bar(counts.index, counts.values)
plt.xlabel('Days Rented')
plt.ylabel('Count')
plt.title('Rental Duration Distribution')
plt.show()

In [ ]:
# Create two columns of dummy variables from "special_features", which takes the value of 1 when:
# The value is "Deleted Scenes", storing as a column called "deleted_scenes".
# The value is "Behind the Scenes", storing as a column called "behind_the_scenes".

#print(original_df['special_features'].value_counts())

original_df['deleted_scenes'] = np.where(original_df['special_features'].str.contains('Deleted Scenes'), 1, 0)
original_df['behind_the_scenes'] = np.where(original_df['special_features'].str.contains('Behind the Scenes'), 1, 0)

In [ ]:
print(original_df['deleted_scenes'].value_counts())
print(original_df['behind_the_scenes'].value_counts())

In [ ]:
y = original_df['rental_length_days'].values
X = original_df
X = X.drop(['rental_date','return_date','rental_length_days','special_features'],axis=1)

In [ ]:
print(type(y))
#print(y.dtypes)
print(X.info())

In [ ]:
# now split the data
SEED=9

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=SEED)

In [ ]:
lasso = Lasso(alpha=0.01)
lasso_coef = lasso.fit(X, y).coef_

plt.bar(X.columns, lasso_coef)
plt.xticks(rotation=90)
plt.figure(figsize=(20,6))
plt.show()

In [ ]:
scores = []
for alpha in [0.001, 0.01, 0.1, 1.0, 10.0, 100.0, 1000.0]:
   lasso = Lasso(alpha=alpha)
   lasso.fit(X_train, y_train )
   lasso_pred = lasso.predict ( X_test)
   scores.append (lasso.score (X_test, y_test ))

print (scores)

In [ ]:
# Instantiate rf
rf = RandomForestRegressor(n_estimators=200,
            random_state=SEED)
           
# Fit rf to the training set    
rf.fit(X_train, y_train)

# Import mean_squared_error as MSE
from sklearn.metrics import mean_squared_error as MSE

# Predict the test set labels
y_pred = rf.predict(X_test)

# Evaluate the test set RMSE
rmse_test = MSE(y_test,y_pred)**0.5

# Print rmse_test
print('Test set RMSE of rf: {:.2f}'.format(rmse_test))

# Create a pd.Series of features importances
importances = pd.Series(data=rf.feature_importances_,
                        index= X_train.columns)

# Sort importances
importances_sorted = importances.sort_values()

# Draw a horizontal barplot of importances_sorted
importances_sorted.plot(kind='barh', color='lightgreen')
plt.title('Features Importances')
plt.show()

In [ ]:
sns.kdeplot(data=original_df, x='length', y='rental_length_days', fill=True)
plt.show()
plt.hexbin(original_df['length'], original_df['rental_length_days'], gridsize=30, cmap='YlOrRd')
plt.colorbar(label='count')
plt.show()

In [ ]:
#original_df.plot(kind='scatter',x='length_2',y='rental_length_days')
#plt.show()
sns.kdeplot(data=original_df, x='length_2', y='rental_length_days', fill=True)
plt.show()
plt.hexbin(original_df['length_2'], original_df['rental_length_days'], gridsize=30, cmap='YlOrRd')
plt.colorbar(label='count')
plt.show()
#sns.regplot(data=original_df,x='length',y='rental_length_days',fitreg=True)

In [ ]:
# Calculate the Pearson correlation of xs and ys

correlation, pvalue = pearsonr(original_df['length_2'], original_df['rental_length_days'])

print(correlation)

In [ ]:
# now drop length_2

X_new = X.drop(['length_2','amount','amount_2'],axis=1)

In [ ]:
# now split the data
SEED=9

X_train, X_test, y_train, y_test = train_test_split(X_new, y, test_size=0.2, random_state=SEED)

In [ ]:
# Instantiate rf
rf = RandomForestRegressor(n_estimators=200,
            random_state=SEED)
           
# Fit rf to the training set    
rf.fit(X_train, y_train)

# Import mean_squared_error as MSE
from sklearn.metrics import mean_squared_error as MSE

# Predict the test set labels
y_pred = rf.predict(X_test)

# Evaluate the test set RMSE
mse_test = MSE(y_test,y_pred)
rmse_test = mse_test**0.5

# Print results
print('Test set MSE of rf: {:.2f}'.format(mse_test))
print('Test set RMSE of rf: {:.2f}'.format(rmse_test))

# Create a pd.Series of features importances
importances = pd.Series(data=rf.feature_importances_,
                        index= X_train.columns)

# Sort importances
importances_sorted = importances.sort_values()

# Draw a horizontal barplot of importances_sorted
importances_sorted.plot(kind='barh', color='lightgreen')
plt.title('Features Importances')
plt.show()

In [ ]:
# using a none hyperparameter tuned model yields an MSE of 2.72 which already exceeds the target MSE of 3.0.

# Hyperparameter tuning at this point is to try and explore if any meaningful improvements can be acheived

params_rf = {
    'max_depth':[3,6,9,12,16, None],
    'min_samples_leaf':[0.02, 0.05, 0.1,0.2],
    'max_features':['0.5','sqrt'],
    'n_estimators':[100,200,300] 
    }


rf = RandomForestRegressor(random_state=SEED)

In [ ]:
# Instantiate grid_dt
grid_rf = RandomizedSearchCV(estimator=rf,
                       param_distributions=params_rf,
                       scoring='neg_mean_squared_error',
                       cv=6,
                       verbose=1,
                       n_jobs=-1)

# Fit grid_dt
grid_rf.fit(X_train,y_train)

#Extarct the best hyperparameters
print('Best hyperparameters:\n', grid_rf.best_params_)

# Extract the best estimator
best_model = grid_rf.best_estimator_

# Predict the test set probabilities of the positive class
y_pred = best_model.predict(X_test)

# Compute MSE
mse = MSE(y_test,y_pred)

# Compute RMSE
rmse = mse**(1/2)

# Print test_roc_auc
print('Test set MSE score: {:.3f}'.format(mse))
print('Test set RMSE score: {:.3f}'.format(rmse))

In [ ]:
# Create a pd.Series of features importances
importances = pd.Series(data=best_model.feature_importances_,
                        index= X_train.columns)

# Sort importances
importances_sorted = importances.sort_values()

# Draw a horizontal barplot of importances_sorted
importances_sorted.plot(kind='barh', color='lightgreen')
plt.title('Features Importances')
plt.show()

In [ ]:
X_final = X_new.drop(['rental_rate_2','PG-13','NC-17','R','PG'],axis=1)

# now split the data
SEED=9

X_train, X_test, y_train, y_test = train_test_split(X_final, y, test_size=0.2, random_state=SEED)

In [ ]:
params_rf = {     'max_depth':[3,6,9,12,16, None],     'min_samples_leaf':[0.02, 0.05, 0.1,0.2],     'max_features':['0.5','sqrt'],     'n_estimators':[100,200,300]      } 
rf = RandomForestRegressor(random_state=SEED)

# Instantiate grid_dt
grid_rf = RandomizedSearchCV(estimator=rf,
                       param_distributions=params_rf,
                       scoring='neg_mean_squared_error',
                       cv=6,
                       verbose=1,
                       n_jobs=-1)

# Fit grid_dt
grid_rf.fit(X_train,y_train)

#Extarct the best hyperparameters
print('Best hyperparameters:\n', grid_rf.best_params_)

# Extract the best estimator
best_model = grid_rf.best_estimator_

# Predict the test set probabilities of the positive class
y_pred = best_model.predict(X_test)

# Compute MSE
best_mse = MSE(y_test,y_pred)

# Compute RMSE
rmse = mse**(1/2)

# Print test_roc_auc
print('Test set MSE score: {:.3f}'.format(mse))
print('Test set RMSE score: {:.3f}'.format(rmse))

# Create a pd.Series of features importances
importances = pd.Series(data=best_model.feature_importances_,
                        index= X_train.columns)

# Sort importances
importances_sorted = importances.sort_values()

# Draw a horizontal barplot of importances_sorted
importances_sorted.plot(kind='barh', color='lightgreen')
plt.title('Features Importances')
plt.show()

In [ ]:
print(best_model)
print(best_mse)

In [ ]:
from sklearn.linear_model import Ridge
from sklearn.model_selection import GridSearchCV

ridge = Ridge()

param_grid = {'alpha': [0.01, 0.1, 1.0, 10, 100, 1000]}

search = GridSearchCV(ridge, param_grid, scoring='neg_mean_squared_error', cv=5)
search.fit(X_train, y_train)

best_alpha = search.best_params_

ridge_final  = Ridge(alpha=best_alpha)

ridge_model = ridge.fit(X_train,y_train)

# Predict the test set probabilities of the positive class
y_pred = ridge_model.predict(X_test)

# Compute MSE
Ridge_best_mse = MSE(y_test,y_pred)

# Print test_roc_auc
print('Test set MSE score: {:.3f}'.format(Ridge_best_mse))
print('Test set RMSE score: {:.3f}'.format(Ridge_best_mse**0.5))